# Create Scenes

(advanced:create-scenes)=

This tutorial demonstrates basic usage around writing and reading [Ngff Scenes](https://ngff.openmicroscopy.org/specifications/dev/index.html#scene-metadata) using the {py:class}`ome_zarr.classes.scene.OMEZarrScene` class.

In [1]:
from skimage import data

from ome_zarr import OMEZarrImage, OMEZarrMultiscale, OMEZarrScene

We create some sample data, which we will store as a tiled layout in a scene zarr group:

In [2]:
example_image = data.human_mitosis()
example_image.shape

c:\Users\johan\Documents\GitHub\ome-zarr-py\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(512, 512)

First, we cut the image into four tiles and convert them into instances of {py:class}`ome_zarr.classes.image.OMEZarrMultiscale`:

In [3]:
img1 = OMEZarrImage(data=example_image[:256, :256], axes=["y", "x"], name="img1")
img2 = OMEZarrImage(data=example_image[256:, :256], axes=["y", "x"], name="img2")
img3 = OMEZarrImage(data=example_image[:256, 256:], axes=["y", "x"], name="img3")
img4 = OMEZarrImage(data=example_image[256:, 256:], axes=["y", "x"], name="img4")

img1_ms = OMEZarrMultiscale(img1)
img2_ms = OMEZarrMultiscale(img2)
img3_ms = OMEZarrMultiscale(img3)
img4_ms = OMEZarrMultiscale(img4)

Next, we need to define a coordinate system into which all images are projected.
This is defined in accordance with the NGFF [coordinate systems specification](https://ngff.openmicroscopy.org/specifications/dev/index.html#coordinatesystems-metadata).

In [4]:
coordinate_system = {
    "name": "world",
    "axes": [
        {"name": "y", "type": "space"},
        {"name": "x", "type": "space"},
    ],
}

We then define translations that move each tile into the appropriate position in the world coordinate system.
In this example, these are simple translations in the y and x dimensions:

In [5]:
coordinate_transformations = [
    {
        "type": "translation",
        "translation": [0, 0],
        "input": {"path": "img1", "name": "physical"},
        "output": {"name": "world"},
    },
    {
        "type": "translation",
        "translation": [256, 0],
        "input": {"path": "img2", "name": "physical"},
        "output": {"name": "world"},
    },
    {
        "type": "translation",
        "translation": [0, 256],
        "input": {"path": "img3", "name": "physical"},
        "output": {"name": "world"},
    },
    {
        "type": "translation",
        "translation": [256, 256],
        "input": {"path": "img4", "name": "physical"},
        "output": {"name": "world"},
    },
]

We can then create and write a scene like this:

In [6]:
scene = OMEZarrScene(
    images=[img1_ms, img2_ms, img3_ms, img4_ms],
    coordinate_systems=[coordinate_system],
    coordinate_transformations=coordinate_transformations
)

scene.to_ome_zarr("test_example_scene.zarr", overwrite=True)

Writing images: 100%|██████████| 4/4 [00:00<00:00,  7.08it/s]


....and load it back like this:

In [12]:
loaded_scene = OMEZarrScene.from_ome_zarr("test_example_scene.zarr")
loaded_scene.coordinate_transformations

(Translation(type='translation', input=CoordinateSystemIdentifier(name='physical', path='img1'), output=CoordinateSystemIdentifier(name='world', path=None), name=None, translation=(0.0, 0.0)),
 Translation(type='translation', input=CoordinateSystemIdentifier(name='physical', path='img2'), output=CoordinateSystemIdentifier(name='world', path=None), name=None, translation=(256.0, 0.0)),
 Translation(type='translation', input=CoordinateSystemIdentifier(name='physical', path='img3'), output=CoordinateSystemIdentifier(name='world', path=None), name=None, translation=(0.0, 256.0)),
 Translation(type='translation', input=CoordinateSystemIdentifier(name='physical', path='img4'), output=CoordinateSystemIdentifier(name='world', path=None), name=None, translation=(256.0, 256.0)))